# Autism Screening Prediction (AQ-10)

Predicts whether someone should be referred for a full autism assessment, from their **AQ-10 Adult** screening answers plus a few background fields.

**This is a screening aid, not a diagnosis.**

Pipeline:
1. Load and clean the Kaggle *Autism Prediction* data (`?` → missing).
2. Explore the data.
3. Hold out a 20% test set.
4. Compare models with 5-fold stratified cross-validation:
   * the questionnaire's own rule (AQ-10 total ≥ 6), as the baseline
   * Logistic Regression
   * Random Forest
5. Evaluate the chosen model once on the test set.

All preprocessing (imputation, scaling, one-hot encoding) runs inside scikit-learn Pipelines, so it is fitted on training folds only. The same logic is in `autism_model.py` / `train.py`, so this notebook and the scripts always agree.

In [ ]:
# Running in Google Colab? Uncomment these lines first:
# !git clone https://github.com/aastha-m22/Autism-Prediction-System.git
# %cd Autism-Prediction-System
# !pip install -q -r requirements.txt
# then upload Kaggle's train.csv into the data/ folder (Files panel on the left)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
                             classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split

from autism_model import (AQ_CUTOFF, AQ_ITEMS, SCORING, SEED, TARGET, aq10_total,
                          build_models, load_data, run_cv, split_xy)

%matplotlib inline

sns.set_theme(style="whitegrid")
DATA_PATH = "data/train.csv"
WITH_DEMOGRAPHICS = False  # set True to also use ethnicity and country

## 1. Load and clean

In [ ]:
df = load_data(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.isna().sum()[lambda s: s > 0]

## 2. Explore

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x=TARGET, data=df, ax=axes[0]).set_title("Class balance")
sns.histplot(df["age"], bins=30, kde=True, ax=axes[1]).set_title("Age")
sns.countplot(x="gender", hue=TARGET, data=df, ax=axes[2]).set_title("Gender vs class")
plt.tight_layout()

In [ ]:
# AQ-10 total vs outcome: how well does the questionnaire alone separate the classes?
df["aq10_total"] = aq10_total(df)
plt.figure(figsize=(8, 4))
sns.countplot(x="aq10_total", hue=TARGET, data=df)
plt.axvline(AQ_CUTOFF - 0.5, color="red", linestyle="--", label=f"cut-off ({AQ_CUTOFF})")
plt.legend(); plt.title("AQ-10 total by class")
df = df.drop(columns="aq10_total")

In [ ]:
# Share of people scoring 1 on each item, by class
df.groupby(TARGET)[AQ_ITEMS].mean().T.plot.bar(figsize=(10, 4), title="Item endorsement rate by class")

## 3. Train / test split

In [ ]:
X, y = split_xy(df, WITH_DEMOGRAPHICS)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(y_train.value_counts(), y_test.value_counts(), sep="\n")

## 4. Cross-validated model comparison

Class 1 (ASD) is the minority, so models use class weighting, and we look at **recall** (share of real cases caught) and **PR-AUC** rather than accuracy.

In [ ]:
models = build_models(WITH_DEMOGRAPHICS, random_state=SEED)
cv_results = run_cv(models, X_train, y_train)
cv_results[list(SCORING)].style.format("{:.3f}").highlight_max(axis=0)

## 5. Final evaluation on the held-out test set

In [ ]:
learned = cv_results.drop(index="AQ-10 rule (total >= 6)")
best_name = learned["pr_auc"].idxmax()
model = models[best_name].fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("Selected:", best_name)
print(classification_report(y_test, pred, target_names=["No ASD", "ASD"], digits=3))  # y_true first!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["No ASD", "ASD"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1], name=best_name)
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[2], name=best_name)
axes[2].plot([0, 1], [0, 1], "--", color="grey")
plt.tight_layout()

Permutation importance, measured on the test set, so it reflects what the model actually relies on for unseen data.

In [ ]:
imp = permutation_importance(model, X_test, y_test, scoring="average_precision", n_repeats=20, random_state=SEED)
pd.Series(imp.importances_mean, index=X_test.columns).sort_values().plot.barh(figsize=(8, 6))
plt.xlabel("Drop in PR-AUC when shuffled"); plt.title("Feature importance")

## Notes and limitations

* The Kaggle data is **synthetic** (generated from the UCI AQ-10 screening data). That is why ages and `result` are fractional, and why some "adult" rows are very young children. Results won't transfer directly to real clinical populations.
* The AQ-10 Adult questionnaire is meant for people aged 16+. Other AQ-10 versions exist for adolescents and children.
* A positive result means *consider a full assessment*, not *has autism*. Diagnosis needs a qualified clinician.